# Graficos dos resultados (Q1-Q6)

Le os CSVs de `results/` e plota os principais indicadores de cada questao.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = "results" if os.path.isdir("results") else "../results"
def load(name):
    p = os.path.join(RESULTS, name)
    return pd.read_csv(p) if os.path.exists(p) else None
def skip(name):
    print(f"[pulado] {name} ainda nao existe")


## Q1 - Pre-treino continuo: base antes/depois vs instruct (held-out PPL, menor melhor)

In [ ]:
df = load("q1_base_vs_instruct.csv")
if df is None: skip("q1_base_vs_instruct.csv")
else:
    d = df[df.eval_set=="heldout"]
    piv = d.pivot_table(index="params", columns="condition", values="ppl", aggfunc="first")
    order = ["0.6B","1.7B","4B","8B","1.0B"]
    piv = piv.reindex([x for x in order if x in piv.index])
    piv.plot(kind="bar"); plt.ylabel("perplexidade (menor melhor)")
    plt.title("Q1 - base antes/depois vs instruct sem treino"); plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()


## Q1 - Ablacao: podar licitacoes do corpus de treino (PPL)

In [ ]:
df = load("q1_balanceamento_licitacao.csv")
if df is None: skip("q1_balanceamento_licitacao.csv")
else:
    d = df[df.condition.isin(["depois"])].pivot_table(index="eval_set", columns="train_corpus", values="ppl", aggfunc="first")
    d.plot(kind="bar"); plt.ylabel("PPL (menor melhor)"); plt.title("Q1 - corpus completo vs balanceado")


## Q2 - SFT: nota do juiz (recall) base vs SFT vs SFT-de-Q1

In [ ]:
df = load("q2_sft.csv")
if df is None: skip("q2_sft.csv")
else:
    d = df[df.eval_set=="recall"].copy()
    d["arm"] = d.apply(lambda r: ("base_antes" if r["condition"]=="antes" else f"sft_{r['start']}"), axis=1)
    piv = d.pivot_table(index="params", columns="arm", values="mean_judge", aggfunc="first")
    piv.plot(kind="bar"); plt.ylabel("juiz 0-5 (maior melhor)"); plt.title("Q2 - SFT por modelo")


## Q3 - LoRA vs SFT pleno (juiz, recall)

In [ ]:
df = load("q3_lora.csv")
if df is None: skip("q3_lora.csv")
else:
    d = df[df.eval_set=="recall"].copy()
    d["lbl"] = d["model"]+" ("+d["start"]+")"
    piv = d.pivot_table(index="lbl", columns="method", values="mean_judge", aggfunc="first")
    piv.plot(kind="bar"); plt.ylabel("juiz 0-5"); plt.title("Q3 - LoRA vs full SFT (~1.7% params)")


## Q4 - Destilacao: base vs distill (juiz) e transfer ratio por aluno

In [ ]:
df = load("q4_distill.csv")
if df is None: skip("q4_distill.csv")
else:
    fig, ax = plt.subplots(1,2, figsize=(13,4))
    df.plot(x="student", y=["base_judge","distill_judge","teacher_judge"], kind="bar", ax=ax[0])
    ax[0].set_title("Q4 - juiz: base vs distill vs teacher"); ax[0].set_ylabel("juiz 0-5"); ax[0].tick_params(axis="x", rotation=30)
    df.plot(x="student", y="transfer_ratio", kind="bar", ax=ax[1], color="green", legend=False)
    ax[1].set_title("Q4 - transfer ratio (fracao do gap fechado)"); ax[1].tick_params(axis="x", rotation=30)


## Q5 - RAG: 3 modos x 3 motores (juiz, corpus cheio)

In [ ]:
# valores agregados do results/README.md (juiz fixo 8B)
import pandas as pd
q5 = pd.DataFrame({
 "baseline":     {"Qwen3-8B":1.10, "gemma-3-1b-it":0.67, "gemma-3-1b-pt":0.47},
 "standard":     {"Qwen3-8B":2.70, "gemma-3-1b-it":2.07, "gemma-3-1b-pt":0.73},
 "agentic":      {"Qwen3-8B":2.60, "gemma-3-1b-it":2.23, "gemma-3-1b-pt":0.73},
 "agentic+grafo":{"Qwen3-8B":2.63, "gemma-3-1b-it":2.03, "gemma-3-1b-pt":0.87}})
q5.plot(kind="bar"); plt.ylabel("juiz 0-5"); plt.title("Q5 - contribuicao do RAG por modo e motor")


## Q6 - Guardrails: protecao com vs sem a camada (+ robustez adversarial)

In [ ]:
df = load("q6_guardrails.csv"); adv = load("q6_adversarial.csv")
if df is None: skip("q6_guardrails.csv")
else:
    fig, ax = plt.subplots(1,2, figsize=(13,4))
    df.plot(x="type", y=["rate_without","rate_with"], kind="bar", ax=ax[0])
    ax[0].set_title("Q6 - taxa tratada (benchmark padrao)"); ax[0].set_ylim(0,1.05); ax[0].tick_params(axis="x", rotation=20)
    if adv is not None:
        adv.plot(x="type", y=["rate_with"], kind="bar", ax=ax[1], color="red", legend=False)
        ax[1].set_title("Q6 - ataques PARAFRASEADOS (regex evade)"); ax[1].set_ylim(0,1.05); ax[1].tick_params(axis="x", rotation=20)
    plt.tight_layout(); plt.show(); print(df)


## Diferenciais: Q3 varredura de rank e Q2 curva de dados (rodam no node02)

In [ ]:
rk = load("q3_rank_sweep.csv"); dc = load("q2_data_curve.csv")
fig, ax = plt.subplots(1,2, figsize=(13,4))
if rk is not None:
    rk = rk.copy(); rk["r"] = rk["model"].str.extract(r"r(\d+)").astype(float)
    rk = rk.dropna(subset=["r"]).sort_values("r")
    ax[0].plot(rk["r"], rk["mean_judge"], "o-"); ax[0].set_xscale("log", base=2)
    ax[0].set_xlabel("rank LoRA"); ax[0].set_ylabel("juiz 0-5"); ax[0].set_title("Q3 - qualidade vs rank")
else: ax[0].set_title("q3_rank_sweep.csv ainda nao existe")
if dc is not None:
    dc = dc.copy(); dc["n"] = dc["model"].str.extract(r"n(\d+)").astype(float)
    dc = dc.dropna(subset=["n"]).sort_values("n")
    ax[1].plot(dc["n"], dc["mean_judge"], "o-", color="purple")
    ax[1].set_xlabel("numero de pares de treino"); ax[1].set_ylabel("juiz 0-5"); ax[1].set_title("Q2 - qualidade vs dados")
else: ax[1].set_title("q2_data_curve.csv ainda nao existe")
